In [ ]:
import os.path
from pathlib import Path
import base64
from email.message import EmailMessage

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# If modifying these scopes, delete the file token_send.json.
# SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
SCOPES = ["https://www.googleapis.com/auth/gmail.send"]

try:
    # Works when running as a .py script
    CREDENTIALS_DIR = Path(__file__).parent.parent / ".credentials"
except NameError:
    # Falls back when running in a Jupyter notebook
    CREDENTIALS_DIR = Path(os.getcwd()) / ".credentials"

TOKEN_JSON_PATH = CREDENTIALS_DIR / "gmail_send_token.json"


In [ ]:
def get_credentials():
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(TOKEN_JSON_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_JSON_PATH, SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                os.path.join(CREDENTIALS_DIR, "credentials.json"), SCOPES
            )
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(TOKEN_JSON_PATH, "w") as token:
            token.write(creds.to_json())
    return creds

In [ ]:
def gmail_send_message():
    """Create and send an email message.
    Print the returned message id.
    Returns: Message object, including message id

    Load pre-authorized user credentials from the environment.
    for more information on Gmail API, API Python Client, and OAuth2, see:
    https://developers.google.com/gmail/api/quickstart/python
    """

    try:
        creds = get_credentials()
        service = build('gmail_assistant', 'v1', credentials=creds)
        message = EmailMessage()

        message.set_content('Hello! This was sent using Python!')

        message['To'] = 'drrandys@yahoo.com'
        message['From'] = 'drrandys@gmail_assistant.com'
        message['Subject'] = 'Testing sending of email with python'

        # encoded message
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

        create_message = {
            'raw': encoded_message
        }
        # pylint: disable=E1101
        send_message = (service.users().messages().send
                        (userId="me", body=create_message).execute())
        print(f'Message Id: {send_message["id"]}')
    except HttpError as error:
        print(f'An error occurred: {error}')
        send_message = None
    return send_message

gmail_send_message()
